# EXPLORAÇÃO DO DATASET BAIXADO - 10 SUJEITOS

**1) Extração de Metadados**

In [1]:
import os
import glob
import pandas as pd
import mne

def get_task_description(run_number):
    try:
        run = int(run_number)
    except ValueError:
        return "Desconhecido"
        
    if run == 1: return "Linha de base (Baseline), olhos abertos"
    elif run == 2: return "Linha de base (Baseline), olhos fechados"
    elif run in [3, 7, 11]: return "Execução Motora: abrir/fechar punho esquerdo ou direito"
    elif run in [4, 8, 12]: return "Imagética Motora: imaginar abrir/fechar punho esquerdo ou direito"
    elif run in [5, 9, 13]: return "Execução Motora: abrir/fechar ambos os punhos ou ambos os pés"
    elif run in [6, 10, 14]: return "Imagética Motora: imaginar abrir/fechar ambos os punhos ou ambos os pés"
    else: return "Tarefa não mapeada na documentação padrão"

def extract_metadata(base_path):
    search_pattern = os.path.join(base_path, '**', '*.edf')
    edf_files = glob.glob(search_pattern, recursive=True)
    
    if not edf_files:
        print("Nenhum arquivo .edf encontrado. Verifique o caminho.")
        return pd.DataFrame()

    print(f"Encontrados {len(edf_files)} arquivos. Iniciando extração com PRELOAD=TRUE...")
    print("Atenção: Pode demorar e consumir muita memória RAM.")
    
    metadata_list = []

    for filepath in edf_files:
        filename = os.path.basename(filepath)
        
        try:
            subject_id = filename[1:4]
            run_id = filename[5:7]     
            
            raw = mne.io.read_raw_edf(filepath, preload=True)
            
            # Extração de metadados do cabeçalho
            sfreq = raw.info['sfreq']
            n_chan = raw.info['nchan']
            duration = raw.times[-1] if len(raw.times) > 0 else 0
            meas_date = raw.info.get('meas_date', None)
            
            annotations = raw.annotations
            event_types = list(set(annotations.description)) if len(annotations) > 0 else []
            
            metadata_list.append({
                'Nome_do_Arquivo': filename,
                'Sujeito_ID': f"S{subject_id}",
                'Run_ID': run_id,
                'Tarefa_Fisica_ou_Mental': get_task_description(run_id),
                'Taxa_Amostragem_Hz': sfreq,
                'Duracao_Segundos': round(duration, 2),
                'Qtd_Canais': n_chan,
                'Tipos_de_Eventos': ", ".join(event_types),
                'Data_Gravacao': meas_date,
                'Caminho_Completo': filepath
            })
            
            # Libera a memória explicitamente após extrair o que precisa deste arquivo
            # Muito importante já que estamos usando preload=True num loop
            del raw 
            
        except Exception as e:
            print(f"Erro ao processar o arquivo {filename}: {e}")

    return pd.DataFrame(metadata_list)

if __name__ == "__main__":
    # O seu caminho de dados
    caminho_dados = r"C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb"
    
    df_metadados = extract_metadata(caminho_dados)
    
    if not df_metadados.empty:
        df_metadados.sort_values(by=['Sujeito_ID', 'Run_ID'], inplace=True)
        
        output_csv = os.path.join(caminho_dados, "eegbci_metadados_completos.csv")
        df_metadados.to_csv(output_csv, index=False, encoding='utf-8-sig', sep=';')
        
        print("\n=== Extração Concluída com Sucesso! ===")
        print(f"Total de registros: {len(df_metadados)}")
        print(f"Arquivo salvo em: {output_csv}")

Encontrados 140 arquivos. Iniciando extração com PRELOAD=TRUE...
Atenção: Pode demorar e consumir muita memória RAM.
Extracting EDF parameters from C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\S004\S004R01.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...
Extracting EDF parameters from C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\S004\S004R02.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...
Extracting EDF parameters from C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\S004\S004R03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Extracting EDF parameters from C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\S004\S004R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ...

**2) Plotagem de Metadados**

In [3]:
import sys
import subprocess

print("Verificando/Instalando dependências...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nbformat', 'plotly', 'pandas', '--quiet'])

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

Verificando/Instalando dependências...


In [ ]:

# 1. Defina o caminho do arquivo CSV
caminho_csv = r"C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\eegbci_metadados_completos.csv"

# 2. Lê os dados
try:
    df = pd.read_csv(caminho_csv, sep=';')
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado no caminho {caminho_csv}")
    exit()

# FUNÇÃO AUXILIAR 
def criar_celulas(dataframe):
    return dict(
        values=[dataframe[col] for col in dataframe.columns],
        fill_color='aliceblue',
        font=dict(color='black', size=12, family="Arial, sans-serif"),
        align='left',
        height=30
    )

# 3. Cria a figura da tabela com os dados completos inicialmente
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df.columns),
        fill_color='midnightblue',
        font=dict(color='white', size=14, family="Arial, sans-serif"),
        align='center',
        height=40
    ),
    cells=criar_celulas(df)
)])

# 4. LÓGICA DO MENU INTERATIVO (FILTRO POR SUJEITO)
sujeitos_unicos = sorted(df['Sujeito_ID'].unique())
botoes_menu = []

# Primeiro botão: Mostrar todos
botoes_menu.append(
    dict(
        label="Todos os Sujeitos",
        method="restyle",
        args=[{"cells": [criar_celulas(df)]}]
    )
)

# Cria um botão de filtro para cada sujeito encontrado no CSV
for sujeito in sujeitos_unicos:
    df_filtrado = df[df['Sujeito_ID'] == sujeito]
    botoes_menu.append(
        dict(
            label=f"Apenas {sujeito}",
            method="restyle",
            args=[{"cells": [criar_celulas(df_filtrado)]}]
        )
    )

# 5. Ajusta o layout (título, margens e insere o menu na tela)
fig.update_layout(
    title_text="Metadados do Dataset EEGBCI - Filtro Dinâmico",
    title_x=0.5, 
    title_font_size=20,
    margin=dict(l=20, r=20, t=110, b=20), # Margem superior (t) aumentada para caber o menu
    height=700, 
    updatemenus=[
        dict(
            buttons=botoes_menu,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.01, # Alinhamento horizontal do menu à esquerda
            xanchor="left",
            y=1.15, # Posição vertical do menu (acima do cabeçalho da tabela)
            yanchor="top",
            bgcolor="white",
            bordercolor="midnightblue"
        )
    ]
)

# 6. Exibe a tabela interativa embutida no VS Code
print("Gerando tabela interativa no VS Code...")
fig.show(renderer="vscode")

Verificando/Instalando dependências...
Gerando tabela interativa no VS Code...


In [6]:
# Conta a quantidade absoluta
print("--- CONTAGEM ABSOLUTA ---")
print(df['Tarefa_Fisica_ou_Mental'].value_counts())

# Conta a proporção em porcentagem (normalize=True)
print("\n--- PORCENTAGEM (%) ---")
distribuicao = df['Tarefa_Fisica_ou_Mental'].value_counts(normalize=True) * 100
print(distribuicao.round(2))

--- CONTAGEM ABSOLUTA ---
Tarefa_Fisica_ou_Mental
Execução Motora: abrir/fechar punho esquerdo ou direito                    30
Imagética Motora: imaginar abrir/fechar punho esquerdo ou direito          30
Execução Motora: abrir/fechar ambos os punhos ou ambos os pés              30
Imagética Motora: imaginar abrir/fechar ambos os punhos ou ambos os pés    30
Linha de base (Baseline), olhos abertos                                    10
Linha de base (Baseline), olhos fechados                                   10
Name: count, dtype: int64

--- PORCENTAGEM (%) ---
Tarefa_Fisica_ou_Mental
Execução Motora: abrir/fechar punho esquerdo ou direito                    21.43
Imagética Motora: imaginar abrir/fechar punho esquerdo ou direito          21.43
Execução Motora: abrir/fechar ambos os punhos ou ambos os pés              21.43
Imagética Motora: imaginar abrir/fechar ambos os punhos ou ambos os pés    21.43
Linha de base (Baseline), olhos abertos                                     7.14
Li

In [4]:
# 1. Defina o caminho do arquivo CSV
caminho_csv = r"C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\eegbci_metadados_completos.csv"

# 2. Lê os dados
try:
    df = pd.read_csv(caminho_csv, sep=';')
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado no caminho {caminho_csv}")
    exit()

print(f"Dados carregados com sucesso! Total de registros: {len(df)}")


# GRÁFICO 1: Contagem de Arquivos por Tarefa

# Encurta os nomes das tarefas para caber melhor no gráfico
df['Tarefa_Curta'] = df['Tarefa_Fisica_ou_Mental'].str.replace('Execução Motora: ', 'Exec: ')\
                                                  .str.replace('Imagética Motora: ', 'Imag: ')\
                                                  .str.replace('Linha de base \(Baseline\), ', 'Baseline: ', regex=True)

contagem_tarefas = df['Tarefa_Curta'].value_counts().reset_index()
contagem_tarefas.columns = ['Tarefa', 'Quantidade']

fig1 = px.bar(
    contagem_tarefas, 
    x='Quantidade', 
    y='Tarefa', 
    orientation='h', # Gráfico de barras horizontais
    title="Distribuição de Arquivos por Tipo de Tarefa",
    color='Tarefa',
    text='Quantidade'
)
fig1.update_layout(showlegend=False, margin=dict(l=20, r=20, t=50, b=20))
fig1.show(renderer="vscode")



# GRÁFICO 2: Duração das Gravações por Tarefa

fig2 = px.box(
    df, 
    x='Tarefa_Curta', 
    y='Duracao_Segundos', 
    color='Tarefa_Curta',
    title="Distribuição da Duração (em segundos) por Tarefa",
    points="all" # Mostra todos os pontos além da caixa do boxplot
)
fig2.update_layout(
    showlegend=False, 
    xaxis_title="Tipo de Tarefa", 
    yaxis_title="Duração (Segundos)"
)
fig2.show(renderer="vscode")



# GRÁFICO 3: Distribuição dos Tipos de Eventos (T0, T1, T2)

contagem_eventos = df['Tipos_de_Eventos'].value_counts().reset_index()
contagem_eventos.columns = ['Eventos', 'Quantidade']

fig3 = px.pie(
    contagem_eventos, 
    names='Eventos', 
    values='Quantidade', 
    title="Proporção dos Tipos de Eventos nos Arquivos",
    hole=0.4, # Transforma num gráfico de rosca (donut chart)
    color_discrete_sequence=px.colors.sequential.Teal
)
fig3.update_traces(textposition='inside', textinfo='percent+label')
fig3.show(renderer="vscode")

Dados carregados com sucesso! Total de registros: 140


**3) Plotagem dos Dados Brutos**

In [7]:
import pandas as pd
import mne
import random

In [9]:
# Configuração para não exibir logs técnicos do MNE no console
mne.set_log_level('ERROR')

# 1. Carrega o CSV que você criou
caminho_csv = r"C:\Users\tiago\Downloads\Mini_Projeto_2_Tiago\data_eegmmidb\eegbci_metadados_completos.csv"
df = pd.read_csv(caminho_csv, sep=';')

# 2. Escolhe um arquivo aleatório do dataset
arquivo_aleatorio = df.sample(n=1).iloc[0]

caminho_edf = arquivo_aleatorio['Caminho_Completo']
sujeito = arquivo_aleatorio['Sujeito_ID']
tarefa = arquivo_aleatorio['Tarefa_Fisica_ou_Mental']

print(f"--- Carregando arquivo aleatório ---")
print(f"Sujeito: {sujeito}")
print(f"Tarefa: {tarefa}")
print(f"Arquivo: {arquivo_aleatorio['Nome_do_Arquivo']}")

# 3. Carrega o arquivo EDF com MNE
raw = mne.io.read_raw_edf(caminho_edf, preload=True)

# 4. Plota o sinal
# O block=True faz o script esperar a janela fechar para terminar
print("Abrindo janela de visualização interativa...")
raw.plot(
    duration=10,     # Mostra 10 segundos de dado por vez na tela
    n_channels=64,   # Mostra 64 canais simultaneamente (ajustável)
    scalings='auto', # Ajusta a escala das ondas automaticamente
    title=f"EEG - {sujeito} - {tarefa}",
    show=True
)

--- Carregando arquivo aleatório ---
Sujeito: S018
Tarefa: Imagética Motora: imaginar abrir/fechar ambos os punhos ou ambos os pés
Arquivo: S018R14.edf
Abrindo janela de visualização interativa...
